In [1]:
# ============================================================
# Cell 1: Create a stable local environment + install deps
# ============================================================
import sys
import shutil
import subprocess

py_ver = sys.version_info
print(f"Python executable: {sys.executable}")
print(f"Python version: {py_ver.major}.{py_ver.minor}.{py_ver.micro}")

# This notebook is pinned to versions that work best on Python 3.10/3.11.
if not ((py_ver.major == 3) and (py_ver.minor in (10, 11))):
    raise RuntimeError(
        "This notebook currently requires Python 3.10 or 3.11 for pinned packages "
        "(especially torch==2.2.0 and numpy==1.26.4).\n"
        "Create a fresh env and select it as notebook kernel, e.g.\n"
        "  py -3.11 -m venv .venv\n"
        "  .\\.venv\\Scripts\\Activate.ps1\n"
        "  python -m pip install --upgrade pip ipykernel\n"
        "  python -m ipykernel install --user --name gru-pruning-venv --display-name \"Python (gru-pruning-venv)\""
    )

in_venv = (hasattr(sys, "base_prefix") and sys.prefix != sys.base_prefix) or hasattr(sys, "real_prefix")
print(f"Virtual environment active: {in_venv}")
if not in_venv:
    print("WARNING: You are not in a venv/conda env. A dedicated env is strongly recommended.")

# Upgrade pip first for better wheel resolution.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"] )

# Force uninstall existing versions to ensure a clean state
print("Uninstalling conflicting packages...")
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchaudio", "numpy", "speechbrain", "huggingface_hub"])


base_packages = [
    "numpy==1.26.4",
    "speechbrain==1.0.0",
    "huggingface_hub==0.23.0",
]

print("Installing base packages from PyPI...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", *base_packages])

torch_core = ["torch==2.2.0", "torchaudio==2.2.0", "torchvision==0.17.0"]

# Detect NVIDIA driver presence via nvidia-smi.
has_nvidia_driver = shutil.which("nvidia-smi") is not None
gpu_compute_cap = None
if has_nvidia_driver:
    try:
        cc = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            check=False,
        )
        if cc.returncode == 0 and cc.stdout.strip():
            gpu_compute_cap = cc.stdout.strip().splitlines()[0].strip()
            print(f"Detected GPU compute capability: sm_{gpu_compute_cap.replace('.', '')}")
    except Exception:
        pass

# Torch 2.2.0 CUDA wheels do not support sm_120 (RTX 50-series).
if gpu_compute_cap and gpu_compute_cap.startswith("12"):
    print("Detected a very new GPU architecture (sm_120).")
    print("Pinned torch==2.2.0 does not support this CUDA architecture; installing CPU PyTorch to keep notebook stable.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", *torch_core])
elif has_nvidia_driver:
    print("NVIDIA driver detected. Trying CUDA-enabled PyTorch wheels (cu121)...")
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir",
            *torch_core,
            "--index-url", "https://download.pytorch.org/whl/cu121",
        ])
    except subprocess.CalledProcessError:
        print("CUDA wheel install failed; falling back to CPU PyTorch wheels.")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", *torch_core])
else:
    print("No NVIDIA driver detected. Installing CPU PyTorch wheels.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", *torch_core])

print("\nDependency installation complete.")
print("IMPORTANT: Please RESTART the kernel now for the changes to take effect, then rerun all cells.")

Python executable: d:\Machine Learning\.venv\Scripts\python.exe
Python version: 3.10.11
Virtual environment active: True
Uninstalling conflicting packages...
Uninstalling conflicting packages...
Installing base packages from PyPI...
Installing base packages from PyPI...
Detected GPU compute capability: sm_86
NVIDIA driver detected. Trying CUDA-enabled PyTorch wheels (cu121)...
Detected GPU compute capability: sm_86
NVIDIA driver detected. Trying CUDA-enabled PyTorch wheels (cu121)...

Dependency installation complete.
IMPORTANT: Please RESTART the kernel now for the changes to take effect, then rerun all cells.

Dependency installation complete.
IMPORTANT: Please RESTART the kernel now for the changes to take effect, then rerun all cells.


In [1]:
# ============================================================
# Cell 2: Verify versions + CUDA availability
# ============================================================
import subprocess
import torch, torchaudio, speechbrain, numpy

print(f"torch:          {torch.__version__}")
print(f"torchaudio:     {torchaudio.__version__}")
print(f"speechbrain:    {speechbrain.__version__}")
print(f"numpy:          {numpy.__version__}")

print(f"torch.version.cuda: {torch.version.cuda}")
print(f"CUDA available:     {torch.cuda.is_available()}")
print(f"CUDA device count:  {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"CUDA device[0]:     {torch.cuda.get_device_name(0)}")
    print(f"cuDNN enabled:      {torch.backends.cudnn.enabled}")
    print(f"cuDNN version:      {torch.backends.cudnn.version()}")

try:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if smi.returncode == 0 and smi.stdout:
        print("\nNVIDIA-SMI detected:")
        print(smi.stdout.splitlines()[0])
    else:
        print("\nNVIDIA-SMI not available in PATH.")
except Exception:
    print("\nNVIDIA-SMI check failed.")

d:\Machine Learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch:          2.2.0+cu121
torchaudio:     2.2.0+cu121
speechbrain:    1.0.0
numpy:          1.26.4
torch.version.cuda: 12.1
CUDA available:     True
CUDA device count:  1
CUDA device[0]:     NVIDIA GeForce RTX 3060 Laptop GPU
cuDNN enabled:      True
cuDNN version:      8801

NVIDIA-SMI detected:
Thu Mar 26 01:05:45 2026       


In [2]:
# ============================================================
# Cell 2a: Install and configure torchaudio backend
# ============================================================
import sys
import subprocess
import torchaudio

# Install soundfile and required audio libraries for Windows
print("Installing soundfile and required audio libraries...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "soundfile", "--upgrade"])

# Modern torchaudio uses the dispatcher pattern (no need to manually set backend)
# soundfile will be auto-detected if installed
print("\nTorchaudio backend configuration:")
print("  - Modern torchaudio uses automatic dispatcher")
print("  - soundfile is installed and will be used automatically for audio I/O")
print("  - No manual backend setting required (deprecated APIs removed)")

# Verify soundfile is available
try:
    import soundfile as sf
    print(f"\nSoundfile version: {sf.__version__}")
    print("Soundfile is ready for use with torchaudio!")
except ImportError:
    print("ERROR: soundfile not installed. Audio loading may fail.")

Installing soundfile and required audio libraries...

Torchaudio backend configuration:
  - Modern torchaudio uses automatic dispatcher
  - soundfile is installed and will be used automatically for audio I/O
  - No manual backend setting required (deprecated APIs removed)

Soundfile version: 0.13.1
Soundfile is ready for use with torchaudio!


In [3]:
# ============================================================
# Cell 3: Load pretrained model + inspect structure
# ============================================================
import os
from pathlib import Path
from huggingface_hub import snapshot_download
from speechbrain.inference.VAD import VAD

# Set environment variable to prevent symlink creation issues on Windows.
os.environ["HF_HUB_LOCAL_DIR_AUTO_SYMLINK_THRESHOLD"] = "500000000"

# Windows-safe download: materialize real files in local_dir (no symlinks).
local_vad_repo_dir = Path("pretrained_models") / "vad-crdnn"
local_vad_repo_dir.mkdir(parents=True, exist_ok=True)

_ = snapshot_download(
    repo_id="speechbrain/vad-crdnn-libriparty",
    local_dir=str(local_vad_repo_dir),
    local_dir_use_symlinks=False,  # Deprecated, but kept for clarity
    revision="main",
)

vad = VAD.from_hparams(
    source=str(local_vad_repo_dir),
    savedir=str(local_vad_repo_dir),
)

# inspect GRU structure
print("=== GRU structure ===")
gru = vad.mods['model'][1]
print(gru)
for name, param in gru.named_parameters():
    print(f"{name}: {list(param.shape)}")

# inspect DNN structure
print("\n=== DNN structure ===")
for name, module in vad.mods['model'][2].named_modules():
    if hasattr(module, 'weight') and len(module.weight.shape) == 2:
        print(f"{name}: weight{list(module.weight.shape)}")

d:\Machine Learning\.venv\lib\site-packages\huggingface_hub\file_download.py:1194: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 8 files: 100%|██████████| 8/8 [00:00<?, ?it/s]

=== GRU structure ===
GRU(
  (rnn): GRU(320, 32, num_layers=2, batch_first=True, bidirectional=True)
)
rnn.weight_ih_l0: [96, 320]
rnn.weight_hh_l0: [96, 32]
rnn.bias_ih_l0: [96]
rnn.bias_hh_l0: [96]
rnn.weight_ih_l0_reverse: [96, 320]
rnn.weight_hh_l0_reverse: [96, 32]
rnn.bias_ih_l0_reverse: [96]
rnn.bias_hh_l0_reverse: [96]
rnn.weight_ih_l1: [96, 64]
rnn.weight_hh_l1: [96, 32]
rnn.bias_ih_l1: [96]
rnn.bias_hh_l1: [96]
rnn.weight_ih_l1_reverse: [96, 64]
rnn.weight_hh_l1_reverse: [96, 32]
rnn.bias_ih_l1_reverse: [96]
rnn.bias_hh_l1_reverse: [96]

=== DNN structure ===
dnn1.linear.w: weight[16, 64]
dnn2.linear.w: weight[16, 16]
lin.w: weight[1, 16]


In [4]:
# ADD THIS AT THE TOP OF CELL 4, before the function definitions:
from speechbrain.inference.VAD import VAD
from pathlib import Path

local_vad_repo_dir = Path("pretrained_models") / "vad-crdnn"
vad = VAD.from_hparams(
    source=str(local_vad_repo_dir),
    savedir=str(local_vad_repo_dir),
)
print("Fresh model loaded for pruning.")

# ============================================================
# Cell 4: GRU structured pruning + fix DNN input dimension
# ============================================================
import torch
import torch.nn as nn

def get_hidden_importance(weight_ih, weight_hh, hidden_size):
    """Compute importance of each hidden unit using L1 norm across all 3 gates."""
    importance = torch.zeros(hidden_size)
    for g in range(3):
        start = g * hidden_size
        end   = (g + 1) * hidden_size
        importance += torch.norm(weight_ih[start:end], p=1, dim=1)
        importance += torch.norm(weight_hh[start:end], p=1, dim=1)
    return importance


def prune_gru(gru_module, pruning_ratio=0.3):
    """
    Structured pruning on a bidirectional 2-layer GRU.
    Removes least important hidden units from each layer.
    Returns n_keep for downstream dimension fixing.
    """
    rnn = gru_module.rnn
    hidden_size = rnn.hidden_size
    n_keep = max(1, int(hidden_size * (1 - pruning_ratio)))
    print(f"GRU hidden_size: {hidden_size} -> {n_keep} (pruned {hidden_size - n_keep})")

    def gate_rows(keep_idx, hidden_size):
        """Get row indices for all 3 gates corresponding to kept hidden units."""
        rows = []
        for g in range(3):
            rows.append(keep_idx + g * hidden_size)
        return torch.cat(rows).sort().values

    # compute importance for layer 0 (combine forward + reverse)
    imp_l0 = (
        get_hidden_importance(rnn.weight_ih_l0.data, rnn.weight_hh_l0.data, hidden_size) +
        get_hidden_importance(rnn.weight_ih_l0_reverse.data, rnn.weight_hh_l0_reverse.data, hidden_size)
    )
    keep_l0 = torch.argsort(imp_l0, descending=True)[:n_keep]
    keep_l0 = keep_l0.sort().values
    rows_l0  = gate_rows(keep_l0, hidden_size)

    # compute importance for layer 1 (combine forward + reverse)
    imp_l1 = (
        get_hidden_importance(rnn.weight_ih_l1.data, rnn.weight_hh_l1.data, hidden_size) +
        get_hidden_importance(rnn.weight_ih_l1_reverse.data, rnn.weight_hh_l1_reverse.data, hidden_size)
    )
    keep_l1 = torch.argsort(imp_l1, descending=True)[:n_keep]
    keep_l1 = keep_l1.sort().values
    rows_l1  = gate_rows(keep_l1, hidden_size)

    # build new smaller GRU
    new_rnn = nn.GRU(
        input_size=rnn.input_size,
        hidden_size=n_keep,
        num_layers=2,
        batch_first=rnn.batch_first,
        bidirectional=True,
        bias=True
    )

    # copy pruned weights: layer 0 forward
    new_rnn.weight_ih_l0.data = rnn.weight_ih_l0.data[rows_l0]
    new_rnn.weight_hh_l0.data = rnn.weight_hh_l0.data[rows_l0][:, keep_l0]
    new_rnn.bias_ih_l0.data   = rnn.bias_ih_l0.data[rows_l0]
    new_rnn.bias_hh_l0.data   = rnn.bias_hh_l0.data[rows_l0]

    # copy pruned weights: layer 0 reverse
    new_rnn.weight_ih_l0_reverse.data = rnn.weight_ih_l0_reverse.data[rows_l0]
    new_rnn.weight_hh_l0_reverse.data = rnn.weight_hh_l0_reverse.data[rows_l0][:, keep_l0]
    new_rnn.bias_ih_l0_reverse.data   = rnn.bias_ih_l0_reverse.data[rows_l0]
    new_rnn.bias_hh_l0_reverse.data   = rnn.bias_hh_l0_reverse.data[rows_l0]

    # copy pruned weights: layer 1 forward (input cols from both directions of l0)
    new_rnn.weight_ih_l1.data = rnn.weight_ih_l1.data[rows_l1][:, torch.cat([keep_l0, keep_l0 + hidden_size])]
    new_rnn.weight_hh_l1.data = rnn.weight_hh_l1.data[rows_l1][:, keep_l1]
    new_rnn.bias_ih_l1.data   = rnn.bias_ih_l1.data[rows_l1]
    new_rnn.bias_hh_l1.data   = rnn.bias_hh_l1.data[rows_l1]

    # copy pruned weights: layer 1 reverse
    new_rnn.weight_ih_l1_reverse.data = rnn.weight_ih_l1_reverse.data[rows_l1][:, torch.cat([keep_l0, keep_l0 + hidden_size])]
    new_rnn.weight_hh_l1_reverse.data = rnn.weight_hh_l1_reverse.data[rows_l1][:, keep_l1]
    new_rnn.bias_ih_l1_reverse.data   = rnn.bias_ih_l1_reverse.data[rows_l1]
    new_rnn.bias_hh_l1_reverse.data   = rnn.bias_hh_l1_reverse.data[rows_l1]

    gru_module.rnn = new_rnn
    print("GRU pruning complete!")
    return n_keep, keep_l1, hidden_size


def fix_dnn_input(dnn, keep_indices, old_hidden_size):
    """Fix dnn1 input dimension using the actual kept GRU hidden-unit indices."""
    old_linear = dnn.dnn1.linear.w
    # columns in old weight: [fwd_0..fwd_H-1, rev_0..rev_H-1]
    keep_cols = torch.cat([keep_indices, keep_indices + old_hidden_size])
    new_input_size = len(keep_cols)
    new_linear = nn.Linear(new_input_size, old_linear.out_features,
                           bias=old_linear.bias is not None)
    new_linear.weight.data = old_linear.weight.data[:, keep_cols]
    if old_linear.bias is not None:
        new_linear.bias.data = old_linear.bias.data
    dnn.dnn1.linear.w = new_linear
    print(f"DNN input fixed: {old_linear.in_features} -> {new_input_size}")


# run GRU pruning
n_keep, keep_l1, old_hidden = prune_gru(vad.mods['model'][1], pruning_ratio=0.3)
fix_dnn_input(vad.mods['model'][2], keep_l1, old_hidden)

Fresh model loaded for pruning.
GRU hidden_size: 32 -> 22 (pruned 10)
GRU pruning complete!
DNN input fixed: 64 -> 44


In [5]:
# ============================================================
# Cell 5: Compare model size before and after pruning
# ============================================================
import os
import tempfile

def count_total_params(vad_model):
    return sum(p.numel() for p in vad_model.mods.parameters())

def get_model_size_mb(vad_model):
    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as tmp:
        temp_path = tmp.name
    try:
        torch.save(vad_model.mods.state_dict(), temp_path)
        return os.path.getsize(temp_path) / (1024 ** 2)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

params_original = 109744
size_original   = 0.44

params_pruned = count_total_params(vad)
size_pruned   = get_model_size_mb(vad)

print(f"{'Metric':<25} {'Original':>10} {'Pruned':>10}")
print("=" * 47)
print(f"{'Total params':<25} {params_original:>10,} {params_pruned:>10,}")
print(f"{'Model size (MB)':<25} {size_original:>10.2f} {size_pruned:>10.2f}")
print(f"{'Param reduction':<25} {'':>10} {(1-params_pruned/params_original):>9.1%}")
print(f"{'Size reduction':<25} {'':>10} {(1-size_pruned/size_original):>9.1%}")

Metric                      Original     Pruned
Total params                 109,744     77,024
Model size (MB)                 0.44       0.32
Param reduction                          29.8%
Size reduction                           27.7%


In [6]:
# ============================================================
# Cell 6: Configure local dataset path + auto-extract archive
# ============================================================
import os
import tarfile
from pathlib import Path

notebook_dir = Path.cwd()
data_root = notebook_dir / "data"

# Default: place LibriParty.tar.gz in the same folder as this notebook.
archive_path = notebook_dir / "LibriParty.tar.gz"

# Prefer using an already extracted folder if present.
candidate_roots = [
    notebook_dir / "LibriParty",
    data_root / "LibriParty",
    notebook_dir / "data" / "LibriParty",
]

libriparty_root = next((p for p in candidate_roots if p.exists()), None)

if libriparty_root is None and archive_path.exists():
    print(f"Found archive: {archive_path}")
    print("Extracting archive (one-time setup)...")
    notebook_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(notebook_dir)
    libriparty_root = next((p for p in candidate_roots if p.exists()), None)
    print("Extraction complete.")

if libriparty_root is None:
    raise FileNotFoundError(
        "LibriParty dataset not found. Do one of the following:\n"
        "1) Put extracted folder at ./LibriParty, OR\n"
        "2) Put LibriParty.tar.gz next to this notebook and rerun Cell 6."
    )

print(f"Using dataset root: {libriparty_root.resolve()}")
print("Top-level items:")
for item in sorted(os.listdir(libriparty_root)):
    print(" -", item)

Using dataset root: D:\Machine Learning\LibriParty
Top-level items:
 - dataset


In [7]:
# ============================================================
# Cell 7: Load up to 100 sessions from LibriParty
# ============================================================
import os
import torchaudio
import json
import random
import soundfile as sf
import torch

random.seed(42)

def load_session(session_dir, session_name, sr=16000, segment_len=3.0):
    """
    Load a LibriParty session mixture and extract speech/non-speech segments.
    Returns (speech_segments, silence_segments) as lists of tensors.
    """
    wav_path  = os.path.join(session_dir, f"{session_name}_mixture.wav")
    json_path = os.path.join(session_dir, f"{session_name}.json")

    # Use soundfile to load the audio, then convert to a tensor
    wav_data, orig_sr = sf.read(wav_path, dtype='float32')
    wav = torch.from_numpy(wav_data).t() # soundfile returns (frames, channels)

    if orig_sr != sr:
        wav = torchaudio.functional.resample(wav, orig_sr, sr)
    
    if wav.ndim > 1 and wav.shape[0] > 1:
        wav = wav.mean(dim=0)  # mono
    else:
        wav = wav.squeeze()


    with open(json_path, "r") as f:
        data = json.load(f)

    seg_samples = int(segment_len * sr)
    speech_segs  = []
    silence_segs = []

    # collect speech intervals from all speakers
    speech_intervals = []
    for speaker_id, utterances in data.items():
        if speaker_id in ("noises", "background"):
            continue
        for utt in utterances:
            speech_intervals.append((utt["start"], utt["stop"]))

    # extract fixed-length speech segments
    for start, stop in speech_intervals:
        start_s = int(start * sr)
        if int(stop * sr) - start_s >= seg_samples:
            seg = wav[start_s:start_s + seg_samples]
            if seg.shape[0] == seg_samples:
                speech_segs.append(seg.unsqueeze(0))

    # extract silence segments from gaps between speech
    speech_intervals_sorted = sorted(speech_intervals, key=lambda x: x[0])
    prev_stop = 0.0
    for start, stop in speech_intervals_sorted:
        gap_start = int(prev_stop * sr)
        gap_stop  = int(start * sr)
        if gap_stop - gap_start >= seg_samples:
            seg = wav[gap_start:gap_start + seg_samples]
            if seg.shape[0] == seg_samples:
                silence_segs.append(seg.unsqueeze(0))
        prev_stop = stop

    return speech_segs, silence_segs

train_dir = os.path.join(str(libriparty_root), "dataset", "train")
if not os.path.isdir(train_dir):
    raise FileNotFoundError(f"Train directory not found: {train_dir}")

all_sessions = [s for s in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, s))]
if not all_sessions:
    raise RuntimeError(f"No sessions found under: {train_dir}")

n_target = min(100, len(all_sessions))
selected_sessions = random.sample(all_sessions, n_target)

all_speech  = []
all_silence = []

print(f"Loading {n_target} sessions...")
for i, session in enumerate(selected_sessions):
    session_path = os.path.join(train_dir, session)
    s, n = load_session(session_path, session)
    all_speech.extend(s)
    all_silence.extend(n)
    if (i + 1) % 10 == 0 or i + 1 == n_target:
        print(f"  {i+1}/{n_target} | speech: {len(all_speech)}, silence: {len(all_silence)}")

# balance dataset
n_samples        = min(len(all_speech), len(all_silence))
if n_samples == 0:
    raise RuntimeError("Could not extract any balanced speech/silence segments.")

speech_balanced  = random.sample(all_speech, n_samples)
silence_balanced = random.sample(all_silence, n_samples)

print(f"\nTotal speech:  {len(all_speech)}")
print(f"Total silence: {len(all_silence)}")
print(f"Balanced:      {n_samples} + {n_samples}")

Loading 100 sessions...
  10/100 | speech: 176, silence: 86
  20/100 | speech: 341, silence: 168
  30/100 | speech: 511, silence: 256
  40/100 | speech: 679, silence: 341
  50/100 | speech: 851, silence: 423
  60/100 | speech: 1023, silence: 507
  70/100 | speech: 1191, silence: 596
  80/100 | speech: 1363, silence: 677
  90/100 | speech: 1536, silence: 762
  100/100 | speech: 1716, silence: 848

Total speech:  1716
Total silence: 848
Balanced:      848 + 848


In [8]:
# ============================================================
# Cell 8: Fine-tune GRU + DNN on LibriParty
# ============================================================

def extract_features(wav, vad_model):
    """Extract Fbank features using VAD model's built-in feature extractor."""
    with torch.no_grad():
        feats = vad_model.mods.compute_features(wav)
        feats = vad_model.mods.mean_var_norm(feats, torch.ones(1))
    return feats


optimizer = torch.optim.Adam([
    {'params': vad.mods['model'][1].parameters(), 'lr': 1e-4},
    {'params': vad.mods['model'][2].parameters(), 'lr': 5e-4},
])
criterion = nn.BCEWithLogitsLoss()
# CNN must stay in eval mode — its batch norm running stats are pretrained
vad.mods['model'].eval()

n_epochs = 10

# Hold out 20% for validation — train only on the rest
random.seed(42)
all_idx = list(range(n_samples))
random.shuffle(all_idx)
val_count = max(1, int(n_samples * 0.2))
train_idx = all_idx[val_count:]
val_idx = all_idx[:val_count]

for epoch in range(n_epochs):
    total_loss = 0
    correct    = 0
    total      = 0

    indices = list(train_idx)
    random.shuffle(indices)

    for i in indices:
        # --- speech sample (label = 1) ---
        feat_s = extract_features(speech_balanced[i], vad)
        with torch.no_grad():
            cnn_out = vad.mods['model'][0](feat_s)
            cnn_out = cnn_out.reshape(cnn_out.shape[0], cnn_out.shape[1], -1)

        optimizer.zero_grad()
        rnn_out = vad.mods['model'][1](cnn_out)
        if isinstance(rnn_out, tuple):
            rnn_out = rnn_out[0]
        dnn_out = vad.mods['model'][2](rnn_out)
        loss_s  = criterion(dnn_out, torch.ones_like(dnn_out))

        # --- silence sample (label = 0) ---
        feat_n = extract_features(silence_balanced[i], vad)
        with torch.no_grad():
            cnn_out_n = vad.mods['model'][0](feat_n)
            cnn_out_n = cnn_out_n.reshape(cnn_out_n.shape[0], cnn_out_n.shape[1], -1)

        rnn_out_n = vad.mods['model'][1](cnn_out_n)
        if isinstance(rnn_out_n, tuple):
            rnn_out_n = rnn_out_n[0]
        dnn_out_n = vad.mods['model'][2](rnn_out_n)
        loss_n    = criterion(dnn_out_n, torch.zeros_like(dnn_out_n))

        loss = loss_s + loss_n
        loss.backward()
        #gradient clipping for stability (especially important with small dataset and higher lr on DNN)
        torch.nn.utils.clip_grad_norm_(
            list(vad.mods['model'][1].parameters()) +
            list(vad.mods['model'][2].parameters()),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

        pred_s = (torch.sigmoid(dnn_out) > 0.5).float()
        pred_n = (torch.sigmoid(dnn_out_n) > 0.5).float()
        correct += pred_s.sum().item() + (1 - pred_n).sum().item()
        total   += pred_s.numel() + pred_n.numel()

    avg_loss = total_loss / n_samples
    accuracy = correct / total * 100
    print(f"Epoch {epoch+1}/{n_epochs} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%")

print("Fine-tuning complete!")




Epoch 1/10 | Loss: 0.8438 | Accuracy: 91.50%
Epoch 2/10 | Loss: 0.6069 | Accuracy: 93.88%
Epoch 3/10 | Loss: 0.5476 | Accuracy: 94.12%
Epoch 4/10 | Loss: 0.5190 | Accuracy: 94.51%
Epoch 5/10 | Loss: 0.4664 | Accuracy: 94.88%
Epoch 6/10 | Loss: 0.4409 | Accuracy: 95.19%
Epoch 7/10 | Loss: 0.4141 | Accuracy: 95.41%
Epoch 8/10 | Loss: 0.4149 | Accuracy: 95.19%
Epoch 9/10 | Loss: 0.3905 | Accuracy: 95.59%
Epoch 10/10 | Loss: 0.3293 | Accuracy: 96.03%
Fine-tuning complete!


In [ ]:
# ============================================================
# Cell 9: Save fine-tuned pruned model + metadata artifacts
# ============================================================
from pathlib import Path
import json
from datetime import datetime
import torch

artifact_dir = Path("artifacts") / "pruned_finetuned_vad"
artifact_dir.mkdir(parents=True, exist_ok=True)

model_ckpt_path = artifact_dir / "vad_pruned_finetuned_state.pt"
meta_path = artifact_dir / "training_metadata.json"
boundaries_path = artifact_dir / "example_vad_boundaries.txt"

# Save model weights
vad.mods.eval()
torch.save(vad.mods.state_dict(), model_ckpt_path)

# Save metadata for reproducibility
metadata = {
    "saved_utc": datetime.utcnow().isoformat() + "Z",
    "epochs_trained": int(globals().get("n_epochs", 0)),
    "final_train_loss": float(globals().get("avg_loss", float("nan"))),
    "final_train_accuracy_percent": float(globals().get("accuracy", float("nan"))),
    "params_pruned": int(globals().get("params_pruned", -1)),
    "size_pruned_mb": float(globals().get("size_pruned", float("nan"))),
    "example_segments_detected": int(len(boundaries)) if "boundaries" in globals() else None,
    "audio_used": str(audio_path) if "audio_path" in globals() else None,
}
meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

# Save final boundary output, if available
if "boundaries" in globals():
    vad.save_boundaries(
        boundaries,
        save_path=str(boundaries_path),
        print_boundaries=False,
        audio_file=str(audio_path) if "audio_path" in globals() else None,
    )
    print(f"Saved boundaries: {boundaries_path.resolve()}")
else:
    print("No `boundaries` variable found; skipped boundary export.")

print(f"Saved model checkpoint: {model_ckpt_path.resolve()}")
print(f"Saved metadata: {meta_path.resolve()}")

Saved boundaries: D:\Machine Learning\artifacts\pruned_finetuned_vad\example_vad_boundaries.txt
Saved model checkpoint: D:\Machine Learning\artifacts\pruned_finetuned_vad\vad_pruned_finetuned_state.pt
Saved metadata: D:\Machine Learning\artifacts\pruned_finetuned_vad\training_metadata.json


In [ ]:
# ============================================================
# Cell 10: Reload saved checkpoint after restart (sanity check)
# ============================================================
from pathlib import Path
import torch
import torch.nn as nn
from speechbrain.inference.VAD import VAD

artifact_dir = Path("artifacts") / "pruned_finetuned_vad"
model_ckpt_path = artifact_dir / "vad_pruned_finetuned_state.pt"

if not model_ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {model_ckpt_path.resolve()}. Run Cell 11 first.")

if "local_vad_repo_dir" not in globals():
    local_vad_repo_dir = Path("pretrained_models") / "vad-crdnn"

# Load checkpoint first so we can reconstruct pruned architecture size
state_dict = torch.load(model_ckpt_path, map_location="cpu")

# Infer pruned hidden size from checkpoint tensor shape: [3 * hidden, input]
ckpt_key = "model.1.rnn.weight_ih_l0"
if ckpt_key not in state_dict:
    raise KeyError(f"Expected key not found in checkpoint: {ckpt_key}")

pruned_hidden = state_dict[ckpt_key].shape[0] // 3
print(f"Detected pruned hidden size from checkpoint: {pruned_hidden}")

# Build base model then reshape modules to pruned dimensions
vad_reloaded = VAD.from_hparams(
    source=str(local_vad_repo_dir),
    savedir=str(local_vad_repo_dir),
)

old_rnn = vad_reloaded.mods['model'][1].rnn
new_rnn = nn.GRU(
    input_size=old_rnn.input_size,
    hidden_size=pruned_hidden,
    num_layers=old_rnn.num_layers,
    batch_first=old_rnn.batch_first,
    bidirectional=old_rnn.bidirectional,
    bias=True,
)
vad_reloaded.mods['model'][1].rnn = new_rnn

old_linear = vad_reloaded.mods['model'][2].dnn1.linear.w
new_linear = nn.Linear(2 * pruned_hidden, old_linear.out_features, bias=old_linear.bias is not None)
vad_reloaded.mods['model'][2].dnn1.linear.w = new_linear

missing, unexpected = vad_reloaded.mods.load_state_dict(state_dict, strict=False)
vad_reloaded.mods.eval()

print(f"Reloaded checkpoint: {model_ckpt_path.resolve()}")
print(f"Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")
print("Reload complete. Use `vad_reloaded` for inference after notebook restart.")

Detected pruned hidden size from checkpoint: 22
Reloaded checkpoint: D:\Machine Learning\artifacts\pruned_finetuned_vad\vad_pruned_finetuned_state.pt
Missing keys: 0 | Unexpected keys: 0
Reload complete. Use `vad_reloaded` for inference after notebook restart.


In [ ]:
# ============================================================
# Cell 11: Results presentation summary
# ============================================================
import math

final_acc = float(globals().get("accuracy", float("nan")))
final_loss = float(globals().get("avg_loss", float("nan")))
segments_now = int(len(boundaries)) if "boundaries" in globals() else None

print("=== Pruned + Fine-tuned Model Summary ===")
print(f"Train epochs run:        {int(globals().get('n_epochs', 0))}")
print(f"Final train loss:        {final_loss:.4f}" if not math.isnan(final_loss) else "Final train loss:        N/A")
print(f"Final train accuracy:    {final_acc:.2f}%" if not math.isnan(final_acc) else "Final train accuracy:    N/A")
print(f"Pruned parameters:       {int(globals().get('params_pruned', -1)):,}" if globals().get("params_pruned", None) is not None else "Pruned parameters:       N/A")
print(f"Pruned model size (MB):  {float(globals().get('size_pruned', float('nan'))):.3f}" if not math.isnan(float(globals().get('size_pruned', float('nan')))) else "Pruned model size (MB):  N/A")
print(f"Segments on example wav: {segments_now if segments_now is not None else 'N/A'}")

if not math.isnan(final_acc):
    if final_acc >= 85:
        verdict = "Good for a pruned model on balanced train data (well above 50% random baseline)."
    elif final_acc >= 75:
        verdict = "Reasonable, but likely needs tuning/validation for deployment confidence."
    else:
        verdict = "Low for practical use; consider tuning pruning ratio, LR, and data balance."
    print(f"\nQuick verdict: {verdict}")

print("\nArtifacts directory:")
print(" - artifacts/pruned_finetuned_vad/vad_pruned_finetuned_state.pt")
print(" - artifacts/pruned_finetuned_vad/training_metadata.json")
print(" - artifacts/pruned_finetuned_vad/example_vad_boundaries.txt")

=== Pruned + Fine-tuned Model Summary ===
Train epochs run:        10
Final train loss:        0.3293
Final train accuracy:    96.03%
Pruned parameters:       77,024
Pruned model size (MB):  0.318
Segments on example wav: 12

Quick verdict: Good for a pruned model on balanced train data (well above 50% random baseline).

Artifacts directory:
 - artifacts/pruned_finetuned_vad/vad_pruned_finetuned_state.pt
 - artifacts/pruned_finetuned_vad/training_metadata.json
 - artifacts/pruned_finetuned_vad/example_vad_boundaries.txt


In [ ]:
# ============================================================
# Cell 12: Validation split evaluation (Precision / Recall / F1)
# ============================================================
import random
import torch

# You can tune these for your experiment
validation_split = 0.2   # 20% held-out validation
validation_seed = 42
threshold = 0.5

if "speech_balanced" not in globals() or "silence_balanced" not in globals() or "n_samples" not in globals():
    raise RuntimeError("Balanced dataset not found. Run Cell 7 first.")
if "vad" not in globals():
    raise RuntimeError("Model not found. Run Cells 3-4-8 first.")
if "extract_features" not in globals():
    raise RuntimeError("Feature extractor helper not found. Run Cell 8 first.")

if not (0.0 < validation_split < 1.0):
    raise ValueError("validation_split must be between 0 and 1.")

# Build a deterministic held-out split from balanced indices
random.seed(validation_seed)
all_idx = list(range(n_samples))
random.shuffle(all_idx)
val_count = max(1, int(n_samples * validation_split))
val_idx = all_idx[:val_count]

vad.mods['model'].eval()

tp = 0
fp = 0
fn = 0
tn = 0

with torch.no_grad():
    for i in val_idx:
        # Speech sample (true label = 1)
        feat_s = extract_features(speech_balanced[i], vad)
        cnn_out_s = vad.mods['model'][0](feat_s)
        cnn_out_s = cnn_out_s.reshape(cnn_out_s.shape[0], cnn_out_s.shape[1], -1)
        rnn_out_s = vad.mods['model'][1](cnn_out_s)
        if isinstance(rnn_out_s, tuple):
            rnn_out_s = rnn_out_s[0]
        logits_s = vad.mods['model'][2](rnn_out_s)
        pred_s = (torch.sigmoid(logits_s) > threshold)

        tp += int(pred_s.sum().item())
        fn += int((~pred_s).sum().item())

        # Silence sample (true label = 0)
        feat_n = extract_features(silence_balanced[i], vad)
        cnn_out_n = vad.mods['model'][0](feat_n)
        cnn_out_n = cnn_out_n.reshape(cnn_out_n.shape[0], cnn_out_n.shape[1], -1)
        rnn_out_n = vad.mods['model'][1](cnn_out_n)
        if isinstance(rnn_out_n, tuple):
            rnn_out_n = rnn_out_n[0]
        logits_n = vad.mods['model'][2](rnn_out_n)
        pred_n = (torch.sigmoid(logits_n) > threshold)

        fp += int(pred_n.sum().item())
        tn += int((~pred_n).sum().item())

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
accuracy_val = (tp + tn) / (tp + tn + fp + fn)

print("=== Validation Split Metrics ===")
print(f"Validation samples per class: {val_count}")
print(f"Threshold: {threshold:.2f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"Accuracy:  {accuracy_val:.4f}")
print(f"Confusion totals -> TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")

=== Validation Split Metrics ===
Validation samples per class: 169
Threshold: 0.50
Precision: 0.9150
Recall:    0.9720
F1-score:  0.9426
Accuracy:  0.9409
Confusion totals -> TP: 49445, FP: 4593, FN: 1424, TN: 46276


In [ ]:
# ============================================================
# Cell 13: Threshold sweep on validation split (pick best F1)
# ============================================================
import random
import torch

# Configurable sweep settings
validation_split = 0.2
validation_seed = 42
threshold_start = 0.05
threshold_end = 0.95
threshold_step = 0.05

if "speech_balanced" not in globals() or "silence_balanced" not in globals() or "n_samples" not in globals():
    raise RuntimeError("Balanced dataset not found. Run Cell 7 first.")
if "vad" not in globals():
    raise RuntimeError("Model not found. Run Cells 3-4-8 first.")
if "extract_features" not in globals():
    raise RuntimeError("Feature extractor helper not found. Run Cell 8 first.")

if threshold_step <= 0:
    raise ValueError("threshold_step must be > 0.")
if not (0.0 < validation_split < 1.0):
    raise ValueError("validation_split must be between 0 and 1.")

# Deterministic validation split
random.seed(validation_seed)
all_idx = list(range(n_samples))
random.shuffle(all_idx)
val_count = max(1, int(n_samples * validation_split))
val_idx = all_idx[:val_count]

# Cache probabilities to make sweeping thresholds fast
speech_probs = []
silence_probs = []

vad.mods['model'].eval()
with torch.no_grad():
    for i in val_idx:
        feat_s = extract_features(speech_balanced[i], vad)
        cnn_out_s = vad.mods['model'][0](feat_s)
        cnn_out_s = cnn_out_s.reshape(cnn_out_s.shape[0], cnn_out_s.shape[1], -1)
        rnn_out_s = vad.mods['model'][1](cnn_out_s)
        if isinstance(rnn_out_s, tuple):
            rnn_out_s = rnn_out_s[0]
        logits_s = vad.mods['model'][2](rnn_out_s)
        speech_probs.append(torch.sigmoid(logits_s).flatten())

        feat_n = extract_features(silence_balanced[i], vad)
        cnn_out_n = vad.mods['model'][0](feat_n)
        cnn_out_n = cnn_out_n.reshape(cnn_out_n.shape[0], cnn_out_n.shape[1], -1)
        rnn_out_n = vad.mods['model'][1](cnn_out_n)
        if isinstance(rnn_out_n, tuple):
            rnn_out_n = rnn_out_n[0]
        logits_n = vad.mods['model'][2](rnn_out_n)
        silence_probs.append(torch.sigmoid(logits_n).flatten())

all_speech_probs = torch.cat(speech_probs)
all_silence_probs = torch.cat(silence_probs)

thresholds = []
current_t = threshold_start
while current_t <= threshold_end + 1e-12:
    thresholds.append(round(current_t, 10))
    current_t += threshold_step

rows = []
best = None
for thr in thresholds:
    pred_s = all_speech_probs > thr
    pred_n = all_silence_probs > thr

    tp = int(pred_s.sum().item())
    fn = int((~pred_s).sum().item())
    fp = int(pred_n.sum().item())
    tn = int((~pred_n).sum().item())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy_val = (tp + tn) / (tp + tn + fp + fn)

    row = {
        "threshold": float(thr),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "accuracy": float(accuracy_val),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }
    rows.append(row)

    if best is None or row["f1"] > best["f1"]:
        best = row

best_threshold = best["threshold"]
print("=== Threshold Sweep (Validation Split) ===")
print(f"Validation samples per class: {val_count}")
print(f"Threshold range: {threshold_start:.2f} to {threshold_end:.2f} (step {threshold_step:.2f})")
print("\nBest threshold by F1:")
print(f"  threshold: {best['threshold']:.2f}")
print(f"  precision: {best['precision']:.4f}")
print(f"  recall:    {best['recall']:.4f}")
print(f"  f1-score:  {best['f1']:.4f}")
print(f"  accuracy:  {best['accuracy']:.4f}")
print(f"  confusion: TP={best['tp']}, FP={best['fp']}, FN={best['fn']}, TN={best['tn']}")

# Optional: top-5 thresholds by F1 for quick review
top5 = sorted(rows, key=lambda r: r["f1"], reverse=True)[:5]
print("\nTop 5 thresholds by F1:")
for r in top5:
    print(f"  thr={r['threshold']:.2f} | F1={r['f1']:.4f} | P={r['precision']:.4f} | R={r['recall']:.4f} | Acc={r['accuracy']:.4f}")

print(f"\nSuggested threshold for Cell 14: threshold = {best_threshold:.2f}")

=== Threshold Sweep (Validation Split) ===
Validation samples per class: 169
Threshold range: 0.05 to 0.95 (step 0.05)

Best threshold by F1:
  threshold: 0.95
  precision: 0.9276
  recall:    0.9703
  f1-score:  0.9485
  accuracy:  0.9473
  confusion: TP=49358, FP=3854, FN=1511, TN=47015

Top 5 thresholds by F1:
  thr=0.95 | F1=0.9485 | P=0.9276 | R=0.9703 | Acc=0.9473
  thr=0.90 | F1=0.9472 | P=0.9251 | R=0.9704 | Acc=0.9459
  thr=0.85 | F1=0.9462 | P=0.9231 | R=0.9705 | Acc=0.9448
  thr=0.80 | F1=0.9453 | P=0.9213 | R=0.9705 | Acc=0.9438
  thr=0.75 | F1=0.9445 | P=0.9198 | R=0.9706 | Acc=0.9430

Suggested threshold for Cell 14: threshold = 0.95
